# Phase 4: fine-tune transformers for scam detection (Google Colab)

**Data:** L3b = L3 text normalisation + policy A (legitimate promotions and service notices are `safe`).
**Models:** `distilbert-base-uncased` (English) and `google/muril-base-cased` (Indian languages incl. romanised Hinglish).

## How to run
1. Runtime > Change runtime type > **T4 GPU**.
2. Upload `phase4_data.zip` to Google Drive at `MyDrive/scam-detector/phase4_data.zip` (made by `python -m src.training.phase4_pack`).
3. Set `SMOKE_TEST = True` in the config cell and **Run all** (a few minutes, tiny subset). Fix nothing yourself; send the error if any.
4. Set `SMOKE_TEST = False` and **Run all**. If Colab disconnects, reconnect and Run all again: finished stages are skipped and
   training resumes from the last epoch checkpoint on Drive.

## Protocol (fixed before any test number is seen)
* Model input is the masked, lowercased `text` column only.
* Early stopping, model selection and the decision threshold use **real validation rows only**. Synthetic validation rows are
  scored afterwards as **diagnostic only**.
* The threshold is the highest one that keeps recall >= `RECALL_TARGET` on real validation scams (recall first).
* The model to export is chosen by real-validation PR-AUC, and `selection.json` is written **before** the test set is scored.
* The real test set is scored **once per model**; predictions are cached on Drive and reused on re-runs.
* Leave-one-source-out (LOSO) retrains without a source and scores that source, with hyperparameters and threshold frozen.

## 1. Configuration

In [ ]:
# ===================== Configuration (edit here) =====================
SMOKE_TEST = False            # True: tiny subset, 1 epoch, separate output folders. Run this FIRST.
RUN_LOSO = True               # leave-one-source-out retraining (the slow part)
LOSO_MODELS = ["distilbert", "multilingual"]   # drop "multilingual" here if Colab quota runs short
LOSO_SOURCES = ["hf_phishing_texts", "uci_sms_spam", "synthetic_llm"]
LOSO_MAX_EPOCHS = 3           # compute cap for LOSO runs (the main runs use MAX_EPOCHS)

MODELS = globals().get("MODELS_OVERRIDE", {"distilbert": "distilbert-base-uncased",
                                           "multilingual": "google/muril-base-cased"})
MAX_LENGTH = 256
SEED = 42
LEARNING_RATE = 2e-5
MAX_EPOCHS = 6
PATIENCE = 2                  # stop after this many epochs without a validation PR-AUC gain
WARMUP_FRACTION = 0.06
WEIGHT_DECAY = 0.01
BATCH_SIZE = {"distilbert": 32, "multilingual": 16}
GRAD_ACCUM = {"distilbert": 1, "multilingual": 2}
RECALL_TARGET = 0.97          # recall-first decision threshold, chosen on real validation rows
OPERATING_TARGETS = (0.95, 0.97, 0.99)   # validation-only trade-off table
SHORT_CHARS = 300
BOOTSTRAP_N = 1000
DELETE_LAST_CHECKPOINT = True  # frees ~3 GB of Drive per model once training is finished

PUSH_TO_HUB = False           # STAYS False until you tell the assistant "go"
HF_REPO_ID = ""               # e.g. "Ridham115/scam-detector-l3b" (only used when PUSH_TO_HUB)
DRIVE_SUBDIR = "scam-detector"

if SMOKE_TEST:
    MAX_EPOCHS, LOSO_MAX_EPOCHS, BOOTSTRAP_N = 1, 1, 50
    LOSO_SOURCES = LOSO_SOURCES[:1]

## 2. Environment, Drive and data

In [ ]:
import json, math, os, platform, random, re, shutil, subprocess, sys, time, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PINNED = ["transformers==5.17.0", "tokenizers==0.23.2", "onnx==1.23.0", "onnxruntime==1.30.0"]
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINNED], check=True)

import numpy as np
import pandas as pd
import torch
import transformers

print("python", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| numpy", np.__version__, "| pandas", pd.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
print("device:", DEVICE, torch.cuda.get_device_name(0) if USE_AMP else "(no GPU: fine for the smoke test only)")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive") / DRIVE_SUBDIR
    DATA_DIR = Path("/content/phase4_data")
else:                                                    # local dry run
    DRIVE = Path(globals().get("LOCAL_DRIVE_DIR", "./_colab_local/drive"))
    DATA_DIR = Path(globals().get("LOCAL_DATA_DIR", "./_colab_local/data"))
DRIVE.mkdir(parents=True, exist_ok=True)
RUN_ROOT = DRIVE / ("runs_smoke" if SMOKE_TEST else "runs")
OUT_ROOT = DRIVE / ("outputs_smoke" if SMOKE_TEST else "outputs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

ZIP_PATH = DRIVE / "phase4_data.zip"
assert ZIP_PATH.exists(), f"upload phase4_data.zip to {DRIVE} first"
DATA_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(DATA_DIR)


def _stratified_head(df: pd.DataFrame, per_group: int) -> pd.DataFrame:
    """Smoke test only: up to ``per_group`` rows for every (source, label) pair."""
    parts = [g.sample(min(len(g), per_group), random_state=SEED) for _, g in df.groupby(["source", "label"])]
    return pd.concat(parts).reset_index(drop=True)


train = pd.read_parquet(DATA_DIR / "train.parquet")
val = pd.read_parquet(DATA_DIR / "val.parquet")
test = pd.read_parquet(DATA_DIR / "test.parquet")
holdout_path = DATA_DIR / "real_indian_test.parquet"
holdout = pd.read_parquet(holdout_path) if holdout_path.exists() else None
if SMOKE_TEST:
    train, val, test = _stratified_head(train, 30), _stratified_head(val, 12), _stratified_head(test, 12)

assert not test["is_synthetic"].any(), "the test set must be 100% real"
assert "text_raw" not in train.columns, "raw text must not be in the Colab pack"
assert set(train["label"]) == {"safe", "scam"}
val_real = val[~val["is_synthetic"]].reset_index(drop=True)
val_synth = val[val["is_synthetic"]].reset_index(drop=True)
print(f"train {len(train):,} | val real {len(val_real):,} + synthetic {len(val_synth):,} | test {len(test):,} | "
      f"real Indian test: {'not built' if holdout is None else len(holdout)}")
print(train.groupby(["source", "label"]).size().unstack(fill_value=0))

## 3. Helpers (pure functions, no torch)

In [ ]:
# PURE
import json, math
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score


def _ratio(num, den):
    return float(num / den) if den else float("nan")


def metrics(y_true, scores, threshold=0.5):
    """Precision / recall / F1 / FPR / AUCs / confusion counts for the scam class (same conventions as src/training/metrics.py)."""
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)
    pred = (s >= threshold).astype(int)
    tn = int(((y == 0) & (pred == 0)).sum())
    fp = int(((y == 0) & (pred == 1)).sum())
    fn = int(((y == 1) & (pred == 0)).sum())
    tp = int(((y == 1) & (pred == 1)).sum())
    precision, recall = _ratio(tp, tp + fp), _ratio(tp, tp + fn)
    f1 = _ratio(2 * precision * recall, precision + recall) if tp else (0.0 if (tp + fp + fn) else float("nan"))
    if y.sum() == 0:
        precision = f1 = float("nan")
    both = len(np.unique(y)) == 2
    return {"n": int(len(y)), "n_scam": int(y.sum()), "threshold": float(threshold), "precision": precision,
            "recall": recall, "f1": f1, "accuracy": _ratio(tp + tn, len(y)), "fpr": _ratio(fp, fp + tn),
            "roc_auc": float(roc_auc_score(y, s)) if both else float("nan"),
            "pr_auc": float(average_precision_score(y, s)) if both else float("nan"),
            "tn": tn, "fp": fp, "fn": fn, "tp": tp}


def recall_first_threshold(y_true, scores, recall_target=0.97):
    """Highest threshold that still flags at least ``recall_target`` of the scam rows.

    The k-th highest scam score is the cut (k = ceil(target * n_scam)); the returned value is the midpoint between it
    and the next lower distinct score, so it flags exactly the same rows but does not sit on a data point.
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)
    pos = np.sort(s[y == 1])[::-1]
    if len(pos) == 0:
        raise ValueError("need at least one scam row to choose a threshold")
    k = min(max(int(math.ceil(recall_target * len(pos) - 1e-9)), 1), len(pos))
    t = float(pos[k - 1])
    lower = s[s < t]
    return float((t + lower.max()) / 2) if len(lower) else t


def operating_points(y_true, scores, targets, benchmark_mask=None):
    """Validation-only trade-off table: for each recall target, the threshold and the price paid in precision / FPR."""
    rows = []
    for target in targets:
        thr = recall_first_threshold(y_true, scores, target)
        m = metrics(y_true, scores, thr)
        row = {"recall_target": target, "threshold": thr, "precision": m["precision"], "recall": m["recall"],
               "fpr": m["fpr"], "fp": m["fp"], "fn": m["fn"]}
        if benchmark_mask is not None and np.asarray(benchmark_mask).any():
            row["promo_flag_rate"] = float((np.asarray(scores)[np.asarray(benchmark_mask)] >= thr).mean())
        rows.append(row)
    return rows


def bootstrap_ci(y_true, scores, threshold, n_boot=1000, seed=42):
    """95% percentile bootstrap interval (resampling rows) for precision, recall, F1, ROC-AUC and PR-AUC."""
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)
    rng = np.random.default_rng(seed)
    draws = {k: [] for k in ("precision", "recall", "f1", "roc_auc", "pr_auc")}
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        m = metrics(y[idx], s[idx], threshold)
        for k in draws:
            draws[k].append(m[k])
    return {k: [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))] for k, v in draws.items()
            if not np.all(np.isnan(v))}


def group_table(df, scores, by, threshold):
    """One metrics row per value of column ``by`` (NaN where a metric is undefined for a group)."""
    y = (df["label"].to_numpy() == "scam").astype(int)
    s = np.asarray(scores, dtype=float)
    rows = {}
    for group, idx in df.reset_index(drop=True).groupby(by).indices.items():
        rows[group] = metrics(y[idx], s[idx], threshold)
    cols = ["n", "n_scam", "precision", "recall", "f1", "fpr", "roc_auc", "pr_auc", "fp", "fn"]
    return pd.DataFrame.from_dict(rows, orient="index")[cols] if rows else pd.DataFrame(columns=cols)


def train_batches(lengths, batch_size, rng, mega=50):
    """Length-bucketed random batches: shuffle, sort inside chunks of ``batch_size * mega``, then shuffle the batches.

    The number of batches is always ceil(n / batch_size), so the learning-rate schedule length is deterministic.
    """
    lengths = np.asarray(lengths)
    order = rng.permutation(len(lengths))
    batches = []
    for start in range(0, len(order), batch_size * mega):
        chunk = order[start:start + batch_size * mega]
        chunk = chunk[np.argsort(lengths[chunk], kind="stable")]
        batches += [chunk[i:i + batch_size] for i in range(0, len(chunk), batch_size)]
    return [batches[i] for i in rng.permutation(len(batches))]


def error_group(row, short_chars=300):
    """Rule-based pattern label for an error row (needs source, benchmark, raw_chars, text)."""
    text = str(row["text"]).lower()
    if bool(row.get("benchmark", False)):
        return "legitimate promotion / service notice"
    if row["source"] == "hf_phishing_texts" and row["raw_chars"] > 1000:
        return "long email"
    if "<url>" in text:
        return "short message with a link"
    if row["raw_chars"] <= short_chars:
        return "short message, no link"
    return "medium message"


def clean_json(obj):
    """Make results JSON-safe: numpy scalars to Python, NaN/inf to null."""
    if isinstance(obj, dict):
        return {str(k): clean_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [clean_json(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating, float)):
        return None if not np.isfinite(obj) else float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, pd.DataFrame):
        return clean_json(obj.reset_index().to_dict(orient="records"))
    return obj

## 4. Model, training loop and scoring
A plain PyTorch loop (not the HF `Trainer`) so that resume, early stopping, class weights and Drive checkpoints behave the same
on any library version. Checkpoints are written atomically once per epoch; `last.pt` allows resuming, `best.pt` holds the best weights.

In [ ]:
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def prep(text: str) -> str:
    """The only text transformation done here (data are already masked and normalised): lowercase, for every model."""
    return text.lower()


def encode(tok, texts):
    """Token id lists (truncated to MAX_LENGTH, special tokens included, no padding)."""
    return tok([prep(t) for t in texts], truncation=True, max_length=MAX_LENGTH)["input_ids"]


def collate(id_lists, pad_id):
    width = max(len(x) for x in id_lists)
    ids = torch.full((len(id_lists), width), pad_id, dtype=torch.long)
    mask = torch.zeros((len(id_lists), width), dtype=torch.long)
    for i, x in enumerate(id_lists):
        ids[i, :len(x)] = torch.tensor(x)
        mask[i, :len(x)] = 1
    return ids, mask


@torch.no_grad()
def predict_probs(model, id_lists, pad_id, batch_size=64):
    """Scam probability per row (softmax in float32), in the original row order."""
    model.eval()
    order = np.argsort([len(x) for x in id_lists], kind="stable")
    out = np.zeros(len(id_lists), dtype=np.float64)
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        ids, mask = collate([id_lists[i] for i in idx], pad_id)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits
        out[idx] = torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy()
    return out


def _atomic_save(obj, path: Path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def train_model(key, hf_name, train_df, val_df, run_dir: Path, persist=True, max_epochs=MAX_EPOCHS):
    """Fine-tune ``hf_name`` with class-weighted loss and early stopping on ``val_df`` PR-AUC.

    With ``persist`` the run resumes from ``run_dir/last.pt`` after a disconnect and best weights live in ``best.pt``;
    without it (LOSO runs) the best weights are kept in memory. Returns ``(model, tokenizer, info)`` with the best weights loaded.
    """
    run_dir.mkdir(parents=True, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(hf_name)
    model = AutoModelForSequenceClassification.from_pretrained(hf_name, num_labels=2).to(DEVICE)
    pad_id = tok.pad_token_id
    tr_ids, va_ids = encode(tok, train_df["text"].tolist()), encode(tok, val_df["text"].tolist())
    y_tr = (train_df["label"].to_numpy() == "scam").astype(np.int64)
    y_va = (val_df["label"].to_numpy() == "scam").astype(np.int64)
    counts = np.bincount(y_tr, minlength=2)
    weights = len(y_tr) / (2.0 * counts)                     # inverse-frequency class weights
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))
    bs, acc = BATCH_SIZE[key], GRAD_ACCUM[key]
    n_batches = math.ceil(len(tr_ids) / bs)
    total_steps = math.ceil(n_batches / acc) * max_epochs
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    groups = [{"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": WEIGHT_DECAY},
              {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups, lr=LEARNING_RATE)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP_FRACTION * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    state = {"epoch": 0, "best": -1.0, "bad": 0, "history": [], "seconds": 0.0}
    last_path, best_path = run_dir / "last.pt", run_dir / "best.pt"
    best_ram = None
    if persist and last_path.exists():
        ck = torch.load(last_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"]); state = ck["state"]
        print(f"[{key}] resumed after epoch {state['epoch']} (best val PR-AUC {state['best']:.4f})")
    print(f"[{key}] {len(tr_ids):,} train rows, class weights safe/scam = {weights[0]:.2f}/{weights[1]:.2f}, "
          f"batch {bs} x accum {acc}, max {max_epochs} epochs, patience {PATIENCE}")
    while state["epoch"] < max_epochs and state["bad"] < PATIENCE:
        ep, t0 = state["epoch"], time.time()
        model.train()
        rng = np.random.default_rng(SEED + ep)
        batches = train_batches([len(x) for x in tr_ids], bs, rng)
        running = 0.0
        for i, idx in enumerate(batches):
            ids, mask = collate([tr_ids[j] for j in idx], pad_id)
            y = torch.tensor(y_tr[idx], device=DEVICE)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits
            loss = loss_fn(logits.float(), y) / acc
            scaler.scale(loss).backward()
            running += loss.item() * acc
            if (i + 1) % acc == 0 or i + 1 == len(batches):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        p_val = predict_probs(model, va_ids, pad_id)
        ap = float(average_precision_score(y_va, p_val))
        state["history"].append({"epoch": ep + 1, "train_loss": running / len(batches), "val_pr_auc": ap,
                                 "val_roc_auc": float(roc_auc_score(y_va, p_val)), "minutes": (time.time() - t0) / 60})
        state["seconds"] += time.time() - t0
        if ap > state["best"] + 1e-6:
            state["best"], state["bad"] = ap, 0
            sd = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if persist:
                _atomic_save(sd, best_path)
            else:
                best_ram = sd
        else:
            state["bad"] += 1
        state["epoch"] = ep + 1
        h, bad = state["history"][-1], state["bad"]
        note = " (best)" if bad == 0 else f" (no gain {bad}/{PATIENCE})"
        print(f"[{key}] epoch {ep + 1}: train loss {h['train_loss']:.4f} | val PR-AUC {ap:.4f} | {h['minutes']:.1f} min{note}")
        if persist:
            _atomic_save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                          "scaler": scaler.state_dict(), "state": state}, last_path)
    best_sd = torch.load(best_path, map_location="cpu") if persist else best_ram
    model.load_state_dict(best_sd)
    info = {"epochs_run": state["epoch"], "stopped_early": state["bad"] >= PATIENCE, "best_val_pr_auc": state["best"],
            "history": state["history"], "train_seconds": state["seconds"], "n_train": int(len(tr_ids)),
            "class_weights": [float(w) for w in weights], "train_scam_rate": float(y_tr.mean())}
    if persist and DELETE_LAST_CHECKPOINT and last_path.exists():
        last_path.unlink()
    return model, tok, info


def save_hf(model, tok, path: Path):
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(path)
    tok.save_pretrained(path)


def load_hf(path: Path, **kw):
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path, **kw).to(DEVICE)
    return model, tok


def free_gpu():
    import gc
    gc.collect()
    if USE_AMP:
        torch.cuda.empty_cache()


def score_frame(model, tok, df):
    return predict_probs(model, encode(tok, df["text"].tolist()), tok.pad_token_id)

## 5. Train each model and choose its recall-first threshold (real validation rows only)

In [ ]:
STAGE = {}          # per-model results collected across cells
y_val_real = (val_real["label"].to_numpy() == "scam").astype(int)
for key, hf_name in MODELS.items():
    run_dir = RUN_ROOT / key
    sentinel = run_dir / "val_stage.json"
    if sentinel.exists():
        STAGE[key] = json.loads(sentinel.read_text(encoding="utf-8"))
        print(f"[{key}] already trained (sentinel found): val PR-AUC {STAGE[key]['train']['best_val_pr_auc']:.4f}")
        continue
    model, tok, info = train_model(key, hf_name, train, val_real, run_dir)
    p_val = score_frame(model, tok, val_real)
    threshold = recall_first_threshold(y_val_real, p_val, RECALL_TARGET)
    val_m = metrics(y_val_real, p_val, threshold)
    ops = operating_points(y_val_real, p_val, OPERATING_TARGETS, benchmark_mask=val_real["benchmark"].to_numpy())
    np.save(run_dir / "val_real_probs.npy", p_val)
    save_hf(model, tok, run_dir / "hf_model")
    STAGE[key] = clean_json({"hf_name": hf_name, "train": info, "threshold": threshold, "val_real": val_m, "operating_points": ops})
    sentinel.write_text(json.dumps(STAGE[key], indent=1), encoding="utf-8")
    print(f"[{key}] threshold {threshold:.4f} (recall >= {RECALL_TARGET} on real val): "
          f"precision {val_m['precision']:.4f} recall {val_m['recall']:.4f} FPR {val_m['fpr']:.4f}")
    del model
    free_gpu()

### The recall-first trade-off (validation only)
Raising the recall target catches more scams but raises false alarms. A missed scam can cost a victim money; a false alarm on a
legitimate bank OTP or delivery notice teaches users to ignore the app. `RECALL_TARGET` is fixed before test and is **not** tuned on test.

In [ ]:
for key in MODELS:
    print(f"\n{key} ({MODELS[key]}), real validation rows only")
    print(pd.DataFrame(STAGE[key]["operating_points"]).round(4).to_string(index=False))

## 6. Model selection (before the test set is touched)

In [ ]:
best_key = max(MODELS, key=lambda k: STAGE[k]["train"]["best_val_pr_auc"])
selection = {"best_key": best_key, "rule": "highest real-validation PR-AUC, decided before the test set is scored",
             "val_pr_auc": {k: STAGE[k]["train"]["best_val_pr_auc"] for k in MODELS}}
sel_path = RUN_ROOT / "selection.json"
if sel_path.exists():                      # keep the first decision on re-runs
    selection = json.loads(sel_path.read_text(encoding="utf-8"))
    best_key = selection["best_key"]
else:
    sel_path.write_text(json.dumps(selection, indent=1), encoding="utf-8")
print("selected for export:", best_key, selection["val_pr_auc"])

## 7. Score the real test set ONCE per model
Predictions are written to Drive (`id, label, source, prob`, no text). If the file exists it is reused, never re-scored.

In [ ]:
PRED = {}            # key -> dict of DataFrames: test, val_synth, holdout
for key in MODELS:
    run_dir = RUN_ROOT / key
    files = {"test": (OUT_ROOT / f"test_predictions_{key}.csv", test),
             "val_synth": (OUT_ROOT / f"val_synthetic_predictions_{key}.csv", val_synth)}
    if holdout is not None:
        files["holdout"] = (OUT_ROOT / f"real_indian_predictions_{key}.csv", holdout)
    missing = [n for n, (p, _) in files.items() if not p.exists()]
    if missing:
        model, tok = load_hf(run_dir / "hf_model")
        for name in missing:
            path, frame = files[name]
            out = frame[["id", "label", "source"]].copy()
            out["prob"] = score_frame(model, tok, frame) if len(frame) else []
            out.to_csv(path, index=False)
            print(f"[{key}] scored {name} ({len(frame):,} rows) -> {path.name}")
        del model
        free_gpu()
    PRED[key] = {n: pd.read_csv(p) for n, (p, _) in files.items()}
    for name, (_, frame) in files.items():
        assert list(PRED[key][name]["id"]) == list(frame["id"]), f"prediction order mismatch for {name}"

## 8. Evaluation on the real test set (same protocol as Phase 3.5)
Scam is the positive class. The threshold is the recall-first one from validation. Short slice = raw text of at most 300 characters.
The false-positive benchmark is the held-out legitimate promotions and service notices (`benchmark` rows, labelled `safe`).

In [ ]:
EVAL = {}
y_test = (test["label"].to_numpy() == "scam").astype(int)
short_mask = (test["raw_chars"] <= SHORT_CHARS).to_numpy()
bench_mask = test["benchmark"].to_numpy()
for key in MODELS:
    thr = STAGE[key]["threshold"]
    p = PRED[key]["test"]["prob"].to_numpy()
    overall = metrics(y_test, p, thr)
    ev = {"threshold": thr, "overall": overall, "ci95": bootstrap_ci(y_test, p, thr, BOOTSTRAP_N, SEED),
          "short": metrics(y_test[short_mask], p[short_mask], thr),
          "by_source": group_table(test, p, "source", thr),
          "benchmark": {"n": int(bench_mask.sum()), "flag_rate": float((p[bench_mask] >= thr).mean()) if bench_mask.any() else None,
                        "by_source": {s: {"n": int(((test["source"] == s) & bench_mask).sum()),
                                          "flag_rate": float((p[((test["source"] == s) & bench_mask).to_numpy()] >= thr).mean())}
                                      for s in sorted(test.loc[bench_mask, "source"].unique())}},
          "non_promo_safe_fpr": float((p[(y_test == 0) & ~bench_mask] >= thr).mean())}
    # diagnostic only: synthetic validation rows
    ps = PRED[key]["val_synth"]["prob"].to_numpy()
    ev["synthetic_val_diagnostic_only"] = {
        "overall": metrics((val_synth["label"].to_numpy() == "scam").astype(int), ps, thr) if len(val_synth) else None,
        "by_language": group_table(val_synth, ps, "language", thr) if len(val_synth) else None,
        "by_scam_type": group_table(val_synth, ps, "scam_type", thr) if len(val_synth) else None}
    if holdout is not None:
        ph = PRED[key]["holdout"]["prob"].to_numpy()
        ev["real_indian"] = {"overall": metrics((holdout["label"].to_numpy() == "scam").astype(int), ph, thr),
                             "by_language": group_table(holdout, ph, "language", thr)}
    EVAL[key] = ev

summary = pd.DataFrame({k: {**{m: EVAL[k]["overall"][m] for m in ("precision", "recall", "f1", "roc_auc", "pr_auc", "fpr")},
                            "FP": EVAL[k]["overall"]["fp"], "FN": EVAL[k]["overall"]["fn"],
                            "short_F1": EVAL[k]["short"]["f1"], "short_recall": EVAL[k]["short"]["recall"],
                            "promo_flag_rate": EVAL[k]["benchmark"]["flag_rate"]} for k in MODELS}).T
print("REAL TEST SET (recall-first threshold)\n", summary.round(4).to_string())
for key in MODELS:
    print(f"\n[{key}] 95% bootstrap CIs:", {k: [round(x, 4) for x in v] for k, v in EVAL[key]["ci95"].items()})
    print(f"[{key}] by source:\n", EVAL[key]["by_source"].round(4).to_string())
    print(f"[{key}] legitimate promotions/service notices flagged: {EVAL[key]['benchmark']}")
    print(f"[{key}] FPR on all other safe rows: {EVAL[key]['non_promo_safe_fpr']:.4f}")
    d = EVAL[key]["synthetic_val_diagnostic_only"]
    if d["overall"]:
        print(f"[{key}] DIAGNOSTIC ONLY, synthetic validation rows: F1 {d['overall']['f1']:.4f}, recall {d['overall']['recall']:.4f}")
        print(d["by_language"].round(4).to_string())
    if "real_indian" in EVAL[key]:
        print(f"[{key}] real Indian test:", {k: round(v, 4) for k, v in EVAL[key]["real_indian"]["overall"].items() if isinstance(v, float)})
if holdout is None:
    print("\nReal Indian test set: not built, so there is no real Hinglish result.")

## 9. Leave-one-source-out (does the model transfer to a source it never saw?)
Train without the source (early stopping on real validation rows of the *other* sources, hyperparameters frozen, at most
`LOSO_MAX_EPOCHS` epochs), then score that source: its **test** rows, or for `synthetic_llm` (which has no test rows) its validation
rows, labelled diagnostic only. F1 uses the in-distribution threshold; ROC-AUC and PR-AUC do not depend on a threshold.

In [ ]:
LOSO = {}
if RUN_LOSO:
    (RUN_ROOT / "loso").mkdir(exist_ok=True)
    for key in LOSO_MODELS:
        if key not in MODELS:
            continue
        for src in LOSO_SOURCES:
            sentinel = RUN_ROOT / "loso" / f"{key}__{src}.json"
            if sentinel.exists():
                LOSO[(key, src)] = json.loads(sentinel.read_text(encoding="utf-8"))
                continue
            tr = train[train["source"] != src].reset_index(drop=True)
            va = val_real[val_real["source"] != src].reset_index(drop=True)
            ev_df = val_synth if src == "synthetic_llm" else test[test["source"] == src].reset_index(drop=True)
            if len(ev_df) == 0 or ev_df["label"].nunique() < 2:
                print(f"[{key}] LOSO {src}: skipped (needs both classes, has {ev_df['label'].nunique()})")
                continue
            print(f"[{key}] LOSO without {src}: train {len(tr):,}, early-stop val {len(va):,}, score {len(ev_df):,} rows")
            model, tok, info = train_model(key, MODELS[key], tr, va, RUN_ROOT / "loso" / f"{key}__{src}", persist=False,
                                           max_epochs=LOSO_MAX_EPOCHS)
            y = (ev_df["label"].to_numpy() == "scam").astype(int)
            p = score_frame(model, tok, ev_df)
            thr = STAGE[key]["threshold"]
            in_dist = PRED[key]["val_synth"] if src == "synthetic_llm" else PRED[key]["test"]
            in_dist_p = in_dist.set_index("id").loc[ev_df["id"], "prob"].to_numpy()
            res = clean_json({"held_out": src, "diagnostic_only": src == "synthetic_llm", "n": len(ev_df),
                              "loso": metrics(y, p, thr), "in_distribution": metrics(y, in_dist_p, thr),
                              "epochs_run": info["epochs_run"], "best_val_pr_auc": info["best_val_pr_auc"]})
            sentinel.write_text(json.dumps(res, indent=1), encoding="utf-8")
            LOSO[(key, src)] = res
            del model
            free_gpu()
    rows = [{"model": k, "held_out": s, "n": r["n"], "in-dist ROC-AUC": r["in_distribution"]["roc_auc"], "LOSO ROC-AUC": r["loso"]["roc_auc"],
             "LOSO PR-AUC": r["loso"]["pr_auc"], "LOSO F1": r["loso"]["f1"], "LOSO precision": r["loso"]["precision"],
             "LOSO recall": r["loso"]["recall"]} for (k, s), r in LOSO.items()]
    print(pd.DataFrame(rows).round(4).to_string(index=False) if rows else "no LOSO results")
else:
    print("RUN_LOSO is False: skipped")

## 10. Corpus-fingerprint check and mitigation proposal
Trigger (fixed in advance): HF-texts LOSO ROC-AUC below 0.75, or an in-distribution minus LOSO gap above 0.15. If it fires, this cell only
**prints a proposal**; nothing is run. Tell the assistant whether to try it.

In [ ]:
MITIGATION = {"trigger": {"loso_auc_below": 0.75, "gap_above": 0.15}, "fired": {}}
for (key, src), r in LOSO.items():
    if src != "hf_phishing_texts":
        continue
    auc, gap = r["loso"]["roc_auc"], r["in_distribution"]["roc_auc"] - r["loso"]["roc_auc"]
    MITIGATION["fired"][key] = bool(auc < 0.75 or gap > 0.15)
    print(f"[{key}] HF LOSO ROC-AUC {auc:.3f} (in-distribution {r['in_distribution']['roc_auc']:.3f}, gap {gap:.3f}) -> "
          f"{'TRIGGER FIRED' if MITIGATION['fired'][key] else 'trigger not fired'}")
if any(MITIGATION["fired"].values()):
    print("""
PROPOSAL (not run): corpus fingerprints still dominate. Options, in the order I would try them:
  1. Source-adversarial training: add a small head that predicts the data source from the [CLS] vector, trained through a gradient-reversal
     layer, so the encoder is pushed to drop source-identifying features. Caveat: source correlates with label here (India SMS has no scam
     rows, UCI is 96% safe), so removing all source information also removes some real signal; the reversal weight must be tuned on LOSO
     validation, not on the test set.
  2. Source-balanced sampling per class (cheap; Phase 3 showed re-weighting alone did not help TF-IDF).
  3. Drop the email corpus from training and treat it as auxiliary (product target is SMS / WhatsApp).
Cost: about one extra fine-tuning run per variant plus its LOSO runs. Ask the project owner before running any of it.""")
else:
    print("\nTrigger not fired (or LOSO not run): no mitigation proposed.")

## 11. Error analysis of the selected model (30 test errors, grouped by pattern)
Post-hoc: these errors were seen after the final score. Nothing is changed in the model because of them; any change would be a new
experiment validated on the validation set only, and the test set would no longer be blind. Examples show the **masked** text.

In [ ]:
thr = STAGE[best_key]["threshold"]
p_best = PRED[best_key]["test"]["prob"].to_numpy()
err = test.assign(prob=p_best, pred=(p_best >= thr).astype(int), y=y_test)
err = err[err["pred"] != err["y"]].copy()
err["kind"] = np.where(err["y"] == 1, "false negative (missed scam)", "false positive (false alarm)")
err["pattern"] = err.apply(error_group, axis=1, short_chars=SHORT_CHARS) if len(err) else pd.Series(dtype=str)
picked = pd.concat([g.sample(min(len(g), 15), random_state=SEED) for _, g in err.groupby("kind")])
print(f"{len(err)} errors on {len(test)} test rows; showing {len(picked)} (random, up to 15 per kind)\n")
print(err.groupby(["kind", "pattern"]).size().rename("all errors").to_frame().to_string(), "\n")
ERRORS = []
for (kind, pattern), g in picked.groupby(["kind", "pattern"]):
    print(f"--- {kind} | {pattern} | {len(g)} shown ---")
    for _, r in g.iterrows():
        snippet = re.sub(r"\s+", " ", r["text"])[:220]
        print(f"  [{r['source']}, p={r['prob']:.3f}] {snippet}")
        ERRORS.append({"kind": kind, "pattern": pattern, "source": r["source"], "prob": float(r["prob"]), "text": snippet})

## 12. Export the selected model to ONNX, quantize, compare
`torch.onnx.export` (eager attention) -> `onnxruntime` dynamic int8 quantization. fp32 vs int8: metrics at the **same** threshold on the
full real test set, label agreement, probability drift, file size and CPU latency. Latency is measured on this Colab CPU, not on your machine.

In [ ]:
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic
from tokenizers import Tokenizer

EXPORT_DIR = OUT_ROOT / "final_model"
FP32_DIR = OUT_ROOT / "onnx_fp32"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
FP32_DIR.mkdir(parents=True, exist_ok=True)
export_done = EXPORT_DIR / "export_stage.json"

if export_done.exists():
    EXPORT = json.loads(export_done.read_text(encoding="utf-8"))
    print("export already done:", {k: v for k, v in EXPORT.items() if k != "parity_examples"})
else:
    src_dir = RUN_ROOT / best_key / "hf_model"
    model, tok = load_hf(src_dir, attn_implementation="eager")
    model = model.cpu().eval().float()
    use_tt = "token_type_ids" in tok.model_input_names and model.config.model_type != "distilbert"
    names = ["input_ids", "attention_mask"] + (["token_type_ids"] if use_tt else [])

    class Wrapper(nn.Module):
        def __init__(self, m):
            super().__init__()
            self.m = m

        def forward(self, *args):
            return self.m(**dict(zip(names, args))).logits

    sample = tok([prep("Your account is blocked, verify now"), prep("ok see you at 5")], padding=True, truncation=True,
                 max_length=MAX_LENGTH, return_tensors="pt")
    args = tuple(sample[n] for n in names)
    fp32_path = FP32_DIR / "model.onnx"
    kwargs = dict(input_names=names, output_names=["logits"], opset_version=17,
                  dynamic_axes={**{n: {0: "batch", 1: "seq"} for n in names}, "logits": {0: "batch"}})
    try:
        torch.onnx.export(Wrapper(model), args, str(fp32_path), dynamo=False, **kwargs)
    except TypeError:                          # older torch without the ``dynamo`` argument
        torch.onnx.export(Wrapper(model), args, str(fp32_path), **kwargs)
    int8_path = EXPORT_DIR / "model_quantized.onnx"
    quantize_dynamic(str(fp32_path), str(int8_path), weight_type=QuantType.QInt8)
    print(f"fp32 {fp32_path.stat().st_size / 1e6:.1f} MB -> int8 {int8_path.stat().st_size / 1e6:.1f} MB")

    # standalone tokenizer (what the local predictor uses) and its parity with the HF tokenizer
    tok.save_pretrained(EXPORT_DIR)
    assert (EXPORT_DIR / "tokenizer.json").exists(), "fast tokenizer file missing"
    std = Tokenizer.from_file(str(EXPORT_DIR / "tokenizer.json"))
    std.enable_truncation(max_length=MAX_LENGTH)
    std.enable_padding(pad_id=tok.pad_token_id, pad_token=tok.pad_token)
    parity_rows = test["text"].tolist()[:200]
    mismatches = 0
    for i in range(0, len(parity_rows), 20):
        chunk = [prep(t) for t in parity_rows[i:i + 20]]
        hf = tok(chunk, padding=True, truncation=True, max_length=MAX_LENGTH)
        mine = std.encode_batch(chunk)
        for a, b in zip(hf["input_ids"], mine):
            mismatches += int(list(a) != list(b.ids))
    print(f"tokenizer parity: {mismatches} of {len(parity_rows)} rows differ")
    assert mismatches == 0, "standalone tokenizer differs from the HF tokenizer: do not ship"

    cfg = {"model_name": MODELS[best_key], "max_length": MAX_LENGTH, "threshold": STAGE[best_key]["threshold"], "lowercase": True,
           "labels": ["safe", "scam"], "pad_token_id": int(tok.pad_token_id), "pad_token": tok.pad_token,
           "recall_target": RECALL_TARGET, "onnx_inputs": names}
    (EXPORT_DIR / "inference_config.json").write_text(json.dumps(cfg, indent=1), encoding="utf-8")
    model.save_pretrained(EXPORT_DIR)                   # HF weights next to the ONNX file (this folder is what the Hub gets)

    def make_session(path):
        return ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])

    def onnx_probs(sess, texts, batch_size=32):
        order = np.argsort([len(t) for t in texts], kind="stable")
        out = np.zeros(len(texts))
        for s in range(0, len(order), batch_size):
            idx = order[s:s + batch_size]
            enc = std.encode_batch([prep(texts[i]) for i in idx])
            feed = {"input_ids": np.array([e.ids for e in enc], dtype=np.int64),
                    "attention_mask": np.array([e.attention_mask for e in enc], dtype=np.int64)}
            if use_tt:
                feed["token_type_ids"] = np.zeros_like(feed["input_ids"])
            logits = sess.run(None, feed)[0].astype(np.float64)
            e = np.exp(logits - logits.max(axis=1, keepdims=True))
            out[idx] = (e[:, 1] / e.sum(axis=1))
        return out

    texts = test["text"].tolist()
    s32, s8 = make_session(fp32_path), make_session(int8_path)
    t0 = time.time(); p32 = onnx_probs(s32, texts); sec32 = time.time() - t0
    t0 = time.time(); p8 = onnx_probs(s8, texts); sec8 = time.time() - t0
    thr = STAGE[best_key]["threshold"]
    p_torch = PRED[best_key]["test"]["prob"].to_numpy()

    def one_by_one(sess, n=200, warm=10):
        times = []
        for i, t in enumerate(texts[:n + warm]):
            enc = std.encode_batch([prep(t)])
            feed = {"input_ids": np.array([enc[0].ids], dtype=np.int64), "attention_mask": np.array([enc[0].attention_mask], dtype=np.int64)}
            if use_tt:
                feed["token_type_ids"] = np.zeros_like(feed["input_ids"])
            t0 = time.perf_counter(); sess.run(None, feed); dt = time.perf_counter() - t0
            if i >= warm:
                times.append(dt * 1000)
        return {"median_ms": float(np.median(times)), "p95_ms": float(np.percentile(times, 95))}

    cpu = platform.processor()
    if os.path.exists("/proc/cpuinfo"):
        cpu = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), cpu)
    EXPORT = clean_json({
        "model_key": best_key, "model_name": MODELS[best_key], "threshold": thr, "use_token_type_ids": use_tt,
        "size_mb": {"fp32_onnx": fp32_path.stat().st_size / 1e6, "int8_onnx": int8_path.stat().st_size / 1e6},
        "torch_vs_onnx_fp32": {"max_abs_prob_diff": float(np.abs(p32 - p_torch).max()), "mean_abs_prob_diff": float(np.abs(p32 - p_torch).mean()),
                               "label_agreement": float(((p32 >= thr) == (p_torch >= thr)).mean())},
        "fp32_onnx": metrics(y_test, p32, thr), "int8_onnx": metrics(y_test, p8, thr),
        "int8_vs_fp32": {"max_abs_prob_diff": float(np.abs(p8 - p32).max()), "mean_abs_prob_diff": float(np.abs(p8 - p32).mean()),
                         "label_agreement": float(((p8 >= thr) == (p32 >= thr)).mean())},
        "latency_batch1": {"fp32": one_by_one(s32), "int8": one_by_one(s8)},
        "throughput_seconds_per_1000_msgs": {"fp32": sec32 / len(texts) * 1000, "int8": sec8 / len(texts) * 1000},
        "cpu": cpu, "cpu_count": os.cpu_count(), "onnxruntime": ort.__version__, "tokenizer_parity_rows_checked": len(parity_rows),
        "note": "latency measured on the Colab CPU, not the deployment machine"})
    export_done.write_text(json.dumps(EXPORT, indent=1), encoding="utf-8")

print(json.dumps({k: EXPORT[k] for k in ("size_mb", "torch_vs_onnx_fp32", "int8_vs_fp32", "latency_batch1", "throughput_seconds_per_1000_msgs")}, indent=1))
print("fp32 ONNX:", {k: round(EXPORT["fp32_onnx"][k], 4) for k in ("precision", "recall", "f1", "roc_auc")})
print("int8 ONNX:", {k: round(EXPORT["int8_onnx"][k], 4) for k in ("precision", "recall", "f1", "roc_auc")})

## 13. Results file and model card draft

In [ ]:
# PURE
def build_model_card(res: dict) -> str:
    """Model card text filled only from measured values in ``res`` (the results dict); nothing is hard-coded."""
    key = res["selection"]["best_key"]
    ev, st, ex = res["evaluation"][key], res["models"][key], res["export"]
    o, sh, bm = ev["overall"], ev["short"], ev["benchmark"]

    def f(v, d=3):
        return "n/a" if v is None else f"{v:.{d}f}"

    ci = ev["ci95"]
    loso_lines = []
    for r in res.get("loso", []):
        if r["model"] == key:
            tag = " (diagnostic only)" if r["diagnostic_only"] else ""
            loso_lines.append(f"| {r['held_out']}{tag} | {r['n']} | {f(r['in_distribution']['roc_auc'])} | {f(r['loso']['roc_auc'])} | "
                              f"{f(r['loso']['f1'])} |")
    loso = ("\n".join(["| Held-out source | Rows | ROC-AUC (trained on it) | ROC-AUC (never seen) | F1 (never seen) |", "|---|---:|---:|---:|---:|", *loso_lines])
            if loso_lines else "Leave-one-source-out was not run for this model.")
    indian = res["data"]["real_indian_test_rows"]
    indian_line = (f"A hand-collected real Indian test set with {indian} rows was scored (see the results file)." if indian
                   else "**No real Hinglish / Indian-language test set exists yet**: every statement about Hinglish rests on synthetic "
                        "training data and a diagnostic synthetic validation set only.")
    return f"""---
language:
- en
- hi
license: apache-2.0
base_model: {ex['model_name']}
library_name: transformers
pipeline_tag: text-classification
tags:
- scam-detection
- phishing
- sms
- onnx
---

# Scam / phishing message classifier ({ex['model_name']}, recall-first)

Binary classifier (`safe` vs `scam`) for SMS, WhatsApp and email text, fine-tuned from `{ex['model_name']}`. It is one component of a
portfolio project (scam detector with explanations); its output is a probability and a threshold decision, not legal or financial advice.

## Intended use
* Flag likely scam or phishing messages so a user can be warned. The decision threshold ({f(st['threshold'], 4)}) was chosen to keep recall
  of at least {res['config']['recall_target']} on real validation scams, so it deliberately accepts more false alarms.
* Out of scope: automated blocking or deletion of messages, use as the sole evidence of fraud, languages and scripts not listed below,
  and any use on message types that differ from the training sources (for example modern WhatsApp forwards).

## Training data (configuration "L3b")
Real messages from three public sources plus LLM-generated Indian-style messages, deduplicated (exact and near-duplicate) before splitting.
Train {res['data']['train_rows']:,} / validation {res['data']['val_rows']:,} / test {res['data']['test_rows']:,} rows; the test set is 100% real
data. Text is masked (URLs, phone numbers, OTPs), normalised (quote markers, years, currency) and lowercased. Legitimate promotions and
service notices are labelled `safe`. Sources: UCI SMS Spam Collection, `ealvaradob/phishing-dataset` (mostly email), an Indian SMS set, and
1,580 audited synthetic messages (published as a separate dataset). **Check each source's licence before redistributing anything derived from it.**

## Training
Class-weighted cross-entropy, AdamW (lr {res['config']['learning_rate']}), early stopping on real-validation PR-AUC (patience
{res['config']['patience']}), max length {res['config']['max_length']} tokens, seed {res['config']['seed']}, {st['train']['epochs_run']} epochs run.

## Results (real test set, {res['data']['test_rows']:,} messages, threshold {f(st['threshold'], 4)})
| Metric | Value | 95% bootstrap CI |
|---|---:|---|
| Precision | {f(o['precision'])} | {f(ci['precision'][0])} to {f(ci['precision'][1])} |
| Recall | {f(o['recall'])} | {f(ci['recall'][0])} to {f(ci['recall'][1])} |
| F1 | {f(o['f1'])} | {f(ci['f1'][0])} to {f(ci['f1'][1])} |
| ROC-AUC | {f(o['roc_auc'])} | {f(ci['roc_auc'][0])} to {f(ci['roc_auc'][1])} |

Short messages (raw text up to 300 characters, n={sh['n']}): F1 {f(sh['f1'])}, recall {f(sh['recall'])}.
Legitimate promotions and service notices flagged as scam: {f(bm['flag_rate'])} (n={bm['n']}).

### Does it transfer to sources it never saw? (leave-one-source-out)
{loso}

**Read the in-distribution score with care.** The public corpora differ strongly in style, so a model can score high partly by recognising
the corpus instead of the scam. The leave-one-source-out table is the more honest estimate of how it behaves on new kinds of messages.

## Quantized ONNX
`model_quantized.onnx` (int8 dynamic quantization, {f(ex['size_mb']['int8_onnx'], 1)} MB vs {f(ex['size_mb']['fp32_onnx'], 1)} MB fp32).
On the full test set at the same threshold: fp32 F1 {f(ex['fp32_onnx']['f1'])}, int8 F1 {f(ex['int8_onnx']['f1'])}, label agreement
{f(ex['int8_vs_fp32']['label_agreement'], 4)}. CPU latency for one message (median): fp32 {f(ex['latency_batch1']['fp32']['median_ms'], 1)} ms,
int8 {f(ex['latency_batch1']['int8']['median_ms'], 1)} ms on {ex['cpu']} ({ex['cpu_count']} cores; not your deployment hardware).

## Limitations
* {indian_line}
* The real test data is almost entirely English, and mostly old email and UK-style SMS spam; it says little about current Indian scams.
* Scam definition: fraud is scam; legitimate marketing is not. Borderline cases (aggressive but real promotions) will be misjudged.
* Probabilities are not calibrated; use the threshold, not the raw value as a percentage.
* Single training seed; differences of a fraction of a percentage point are noise.
* Messages are treated as untrusted text and only ever tokenized, but adversarial rewording can evade the model.
"""

In [ ]:
RESULTS = clean_json({
    "smoke_test": SMOKE_TEST,
    "config": {"models": MODELS, "max_length": MAX_LENGTH, "seed": SEED, "learning_rate": LEARNING_RATE, "max_epochs": MAX_EPOCHS,
               "patience": PATIENCE, "recall_target": RECALL_TARGET, "loso_max_epochs": LOSO_MAX_EPOCHS, "short_chars": SHORT_CHARS,
               "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM, "bootstrap_n": BOOTSTRAP_N, "device": DEVICE,
               "torch": torch.__version__, "transformers": transformers.__version__},
    "data": {"train_rows": len(train), "val_rows": len(val), "val_real_rows": len(val_real), "val_synthetic_rows": len(val_synth),
             "test_rows": len(test), "real_indian_test_rows": None if holdout is None else len(holdout),
             "test_scam_rate": float(y_test.mean()), "test_benchmark_rows": int(bench_mask.sum())},
    "selection": selection,
    "models": STAGE,
    "evaluation": EVAL,
    "loso": [{"model": k, **r} for (k, _), r in LOSO.items()],
    "mitigation": MITIGATION,
    "error_analysis": ERRORS,
    "export": EXPORT,
})
(OUT_ROOT / "phase4_results.json").write_text(json.dumps(RESULTS, indent=1), encoding="utf-8")
card = build_model_card(RESULTS)
(EXPORT_DIR / "README.md").write_text(card, encoding="utf-8")
print(card[:1800], "\n...\n")
print("wrote", OUT_ROOT / "phase4_results.json", "and", EXPORT_DIR / "README.md")

## 14. Publish to the Hugging Face Hub (OFF until the project owner says "go")
Review `final_model/README.md` on Drive first. The token is read from Colab Secrets (`HF_TOKEN`); never paste it into a cell.
The repo is created **private**; make it public from the Hub settings after checking it.

In [ ]:
if PUSH_TO_HUB:
    assert HF_REPO_ID, "set HF_REPO_ID first"
    assert not SMOKE_TEST, "never publish a smoke-test model"
    from google.colab import userdata
    from huggingface_hub import HfApi
    api = HfApi(token=userdata.get("HF_TOKEN"))
    api.create_repo(HF_REPO_ID, private=True, exist_ok=True)
    api.upload_folder(folder_path=str(EXPORT_DIR), repo_id=HF_REPO_ID, commit_message="Phase 4 model (L3b)",
                      ignore_patterns=["export_stage.json"])
    print("uploaded (private):", f"https://huggingface.co/{HF_REPO_ID}")
else:
    print("PUSH_TO_HUB is False: nothing uploaded.")

## 15. What to download
From `MyDrive/scam-detector/outputs/`: `phase4_results.json`, `test_predictions_*.csv`, `final_model/` (ONNX, tokenizer, config, model card).
Send them to the assistant, which writes `results.md` from the real numbers only.

In [ ]:
for p in sorted(OUT_ROOT.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(OUT_ROOT)!s:60} {p.stat().st_size / 1e6:9.2f} MB")